# Notebook 04: Pretraining Data Collection

**Time:** 30 minutes  
**Prerequisites:** Notebook 03 complete  
**Goal:** Experience the first stage of the LLM pretraining pipeline by collecting real data

## The Pretraining Pipeline

```
📥 Data Collection  →  🧹 Data Cleaning  →  🔤 Tokenization  →  🏋️ Training
 (this notebook)       (notebook 05)        (notebook 03)       (future class)
```

This notebook covers **three data modalities** used in LLM pretraining:
1. **Web text** — scraped from the internet (biggest source by volume)
2. **PDF/document text** — academic papers, books, technical docs
3. **Transcribed speech** — podcasts, lectures, interviews

> 💡 **Real-world context:** GPT-4 was trained on ~45TB of text including web pages, books, code, and user interactions. LLaMA 4 used 22–40 trillion tokens from 200+ languages. This notebook gives you a taste of how that data was collected.

In [2]:
import os, sys, json, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.data_utils import scrape_arxiv, batch_ocr_pdfs, transcribe_youtube
from src.utils import append_to_reflection

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 04")

✅ Setup complete — ready for Notebook 04


/Users/scottlai/Documents/inferenceAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


---

## 📌 Sidebar: Tesseract OCR

**Tesseract** is an open-source OCR engine developed by Google. It powers the PDF and image text extraction in this notebook.

### Installation

| Platform | Command |
|---|---|
| macOS | `brew install tesseract` |
| Ubuntu/Debian | `sudo apt install tesseract-ocr` |
| Windows | [Download from GitHub releases](https://github.com/tesseract-ocr/tesseract/releases) |

### Page Segmentation Modes (`--psm`)

| PSM | Meaning | Best for |
|---|---|---|
| `--psm 3` | Fully automatic segmentation | Default; most documents |
| `--psm 6` | Assume a single uniform block of text | Academic papers, reports |
| `--psm 11` | Sparse text — find as much text as possible | Screenshots, forms |
| `--psm 13` | Raw line, treat as single text line | Single-line OCR |

### Best Practices

- **Preprocess images**: Convert to grayscale, adjust contrast, remove noise
- **DPI matters**: Use ≥200 DPI for good results (300 DPI is ideal)
- **Specify language**: `pytesseract.image_to_string(img, lang='eng+fra')` for multilingual
- **Custom training**: For specialized fonts, Tesseract can be fine-tuned

```python
import pytesseract
from PIL import Image

image = Image.open('document.png')
text  = pytesseract.image_to_string(image, config='--psm 6')
```

---

## Section 1: Web Scraping — arXiv Paper Abstracts

**Trafilatura** is a Python library for web scraping and HTML cleaning. It's used in production by large ML projects because:
- Automatically extracts main content (ignores navigation, ads, footers)
- Respects `robots.txt`
- Handles many edge cases (paywalls, JavaScript-heavy pages)
- 10–100x faster than BeautifulSoup for clean-text extraction

In [2]:
print("=" * 65)
print("🧪 Experiment 1 (Demo): arXiv Web Scraper")
print("=" * 65)
print()
print("Scraping category: cs.CL (Computation and Language)")
print("Fetching 3 papers as a demo...")
print()

demo_papers = scrape_arxiv(
    category="cs.CL",
    max_papers=3,
    save_path=os.path.join(outputs_dir, 'arxiv_clean_demo.json')
)

print()
if demo_papers:
    print("Sample paper extracted:")
    p = demo_papers[0]
    print(f"  URL:      {p['url']}")
    print(f"  Title:    {p['title'][:80]}...")
    print(f"  Abstract: {p['abstract'][:150]}...")
    print()
    print(f"✅ Demo complete — {len(demo_papers)} papers extracted")

🧪 Experiment 1 (Demo): arXiv Web Scraper

Scraping category: cs.CL (Computation and Language)
Fetching 3 papers as a demo...

Fetching arXiv listing: https://arxiv.org/list/cs.CL/recent
Found 3 abstract pages — fetching...
  [1/3] https://arxiv.org/abs/2604.02324
  [2/3] https://arxiv.org/abs/2604.02319
  [3/3] https://arxiv.org/abs/2604.02209

✓ Collected 3 papers
✓ Saved to: ../outputs/arxiv_clean_demo.json

Sample paper extracted:
  URL:      https://arxiv.org/abs/2604.02324
  Title:    Computer Science > Computation and Language...
  Abstract: [Submitted on 2 Apr 2026] Title:Grounded Token Initialization for New Vocabulary in LMs for Generative Recommendation View PDF HTML (experimental)Abst...

✅ Demo complete — 3 papers extracted


### 🎯 TODO 1: Scrape Your Own Category

In [3]:
# TODO 1: Scrape an arXiv category relevant to your project

# Common categories:
#   cs.CL   — Computation and Language (NLP)
#   cs.LG   — Machine Learning
#   cs.CV   — Computer Vision
#   cs.AI   — Artificial Intelligence
#   stat.ML — Statistics and Machine Learning
#   cs.IR   — Information Retrieval
#   cs.RO   — Robotics
#   q-bio.BM — Bioinformatics and Biomolecules (for bio/med)

MY_CATEGORY = "cs.LG"   # ← Change to your category!
MY_MAX_PAPERS = 10

print("=" * 65)
print(f"🎯 TODO 1: Scraping arXiv category: {MY_CATEGORY}")
print("=" * 65)
print()

my_papers = scrape_arxiv(
    category=MY_CATEGORY,
    max_papers=MY_MAX_PAPERS,
    save_path=os.path.join(outputs_dir, 'arxiv_clean.json')
)

print()
print(f"Results summary:")
print(f"  Category:  {MY_CATEGORY}")
print(f"  Papers:    {len(my_papers)}")
total_chars = sum(len(p.get('raw_text', '')) for p in my_papers)
print(f"  Total chars collected: {total_chars:,}")
print(f"  Est. tokens: {total_chars // 4:,}")

todo1_reflection = """
[YOUR REFLECTION HERE]

- Which category did you choose and why?
- How clean was the extracted text? Any issues with formatting?
- What percentage of papers seem relevant to your project topic?
"""
print()
print(todo1_reflection)

🎯 TODO 1: Scraping arXiv category: cs.LG

Fetching arXiv listing: https://arxiv.org/list/cs.LG/recent
Found 10 abstract pages — fetching...
  [1/10] https://arxiv.org/abs/2604.02322
  [2/10] https://arxiv.org/abs/2604.02309
  [3/10] https://arxiv.org/abs/2604.02292
  [4/10] https://arxiv.org/abs/2604.02288
  [5/10] https://arxiv.org/abs/2604.02270
  [6/10] https://arxiv.org/abs/2604.02268
  [7/10] https://arxiv.org/abs/2604.02260
  [8/10] https://arxiv.org/abs/2604.02250
  [9/10] https://arxiv.org/abs/2604.02215
  [10/10] https://arxiv.org/abs/2604.02206

✓ Collected 10 papers
✓ Saved to: ../outputs/arxiv_clean.json

Results summary:
  Category:  cs.LG
  Papers:    10
  Total chars collected: 20,000
  Est. tokens: 5,000


[YOUR REFLECTION HERE]

- Which category did you choose and why?
- How clean was the extracted text? Any issues with formatting?
- What percentage of papers seem relevant to your project topic?



---

## Section 2: PDF OCR

Many valuable documents exist only as PDFs — scanned books, older papers, government reports.
For these, we need OCR: **convert pages to images → run Tesseract → extract text**.

We also compare **pdfplumber** (for PDFs with an embedded text layer) vs. **Tesseract OCR** (for image-based PDFs).

In [4]:
print("=" * 65)
print("🧪 Experiment 2 (Demo): PDF Text Extraction")
print("=" * 65)
print()

# Method A: pdfplumber (for text-layer PDFs — fast, accurate)
print("Method A: pdfplumber (text-layer PDFs)")
print("-" * 40)
try:
    import pdfplumber
    # Use the Week 2 course PDF as a demo
    demo_pdf = os.path.join(parent_dir, 'Week 2.pdf')
    if os.path.exists(demo_pdf):
        with pdfplumber.open(demo_pdf) as pdf:
            print(f"✅ Opened: Week 2.pdf ({len(pdf.pages)} pages)")
            first_page_text = pdf.pages[0].extract_text() or "(no text layer)"
            print(f"   Page 1 text (first 300 chars):")
            print(f"   {first_page_text[:300]}...")
    else:
        print("   Week 2.pdf not found — try with another PDF in your directory")
except ImportError:
    print("⚠️  pdfplumber not installed: pip install pdfplumber")

print()

# Method B: pdf2image + pytesseract (for image/scanned PDFs)
print("Method B: pdf2image + pytesseract (image-based PDFs)")
print("-" * 40)
try:
    import pytesseract
    import shutil
    if shutil.which('tesseract'):
        print(f"✅ Tesseract available: {pytesseract.get_tesseract_version()}")
        print(f"   Use batch_ocr_pdfs(pdf_paths) in TODO 2 below")
    else:
        print("⚠️  Tesseract binary not found")
        print("   macOS:  brew install tesseract poppler")
        print("   Ubuntu: sudo apt install tesseract-ocr poppler-utils")
except Exception as e:
    print(f"⚠️  {e}")

🧪 Experiment 2 (Demo): PDF Text Extraction

Method A: pdfplumber (text-layer PDFs)
----------------------------------------
✅ Opened: Week 2.pdf (33 pages)
   Page 1 text (first 300 chars):
   DATE July 28, 2025 Week 2 - Machine Learning Engineer in the Generative AI Era
Week 2
LLM Overview
Understanding the Architecture and Training of Large Language Models...

Method B: pdf2image + pytesseract (image-based PDFs)
----------------------------------------
✅ Tesseract available: 5.5.2
   Use batch_ocr_pdfs(pdf_paths) in TODO 2 below


### 🎯 TODO 2: Batch OCR Your PDFs

In [5]:
# TODO 2: Run OCR on 2-3 PDFs of your choice

# Option A: Use the course PDF as practice
# Option B: Download PDFs from your arXiv results above
# Option C: Use any PDF files you have locally

# Provide paths to PDF files (absolute or relative to parent_dir)
my_pdf_paths = [
    # os.path.join(parent_dir, 'Week 2.pdf'),   # Course PDF
    # "/path/to/your/paper.pdf",                # Add your PDFs here
]

print("=" * 65)
print("🎯 TODO 2: Batch PDF OCR")
print("=" * 65)
print()

if my_pdf_paths:
    ocr_outputs = batch_ocr_pdfs(
        pdf_paths=my_pdf_paths,
        output_dir=os.path.join(outputs_dir, 'pdf_ocr'),
        max_pages=2,    # limit to 2 pages per PDF for speed
        tesseract_config='--psm 6'
    )

    if ocr_outputs:
        print(f"\n✅ OCR complete: {len(ocr_outputs)} files saved")
        # Show a snippet of the first output
        with open(ocr_outputs[0], 'r') as f:
            sample = f.read(300)
        print(f"\nSample from first output:\n{sample}...")
else:
    print("⚠️  No PDF paths provided.")
    print("   Add PDF file paths to my_pdf_paths above and re-run.")
    print()
    print("   Quick option: download any paper from https://arxiv.org")
    print("   and add its local path to the list.")

todo2_reflection = """
[YOUR REFLECTION HERE]

- Which PDFs did you OCR?
- How accurate was the OCR output? Any errors or garbled text?
- When would you use pdfplumber vs. Tesseract for a given PDF?
- What cleaning steps would the OCR output need before it could be used as training data?
"""
print()
print(todo2_reflection)

🎯 TODO 2: Batch PDF OCR

⚠️  No PDF paths provided.
   Add PDF file paths to my_pdf_paths above and re-run.

   Quick option: download any paper from https://arxiv.org
   and add its local path to the list.


[YOUR REFLECTION HERE]

- Which PDFs did you OCR?
- How accurate was the OCR output? Any errors or garbled text?
- When would you use pdfplumber vs. Tesseract for a given PDF?
- What cleaning steps would the OCR output need before it could be used as training data?



---

## Section 3: ASR — Transcribing YouTube Audio with faster-whisper

**faster-whisper** is a reimplementation of OpenAI's Whisper model that runs 4× faster on CPU using CTranslate2 quantization. It's the standard choice for production ASR pipelines.

**Model sizes:**

| Size | Parameters | Speed | Use case |
|---|---|---|---|
| `tiny` | 39M | fastest | Quick demos, low accuracy |
| `base` | 74M | fast | Good balance for English |
| `small` | 244M | moderate | Better multilingual |
| `medium` | 769M | slow | Near-human accuracy |
| `large-v3` | 1.5B | slowest | Best quality |

> For this homework, `tiny` is sufficient and will run in 1–3 minutes on CPU.

In [ ]:
print("=" * 65)
print("🧪 Experiment 3 (Demo): ASR with faster-whisper")
print("=" * 65)
print()

# This is a short public domain video — the "Attention is All You Need" paper talk
demo_url = "https://www.youtube.com/watch?v=rBCqOTEfxvg"   # Yannic Kilcher: Attention is All You Need

print(f"Demo video: {demo_url}")
print("Using model: tiny (fastest, ~1-3 min on CPU for first 3 min of audio)")
print()
print("Note: First run downloads the tiny model (~75MB). Subsequent runs use cache.")
print()

## Uncomment to run the demo (takes 2-5 minutes on first run)
# demo_transcript = transcribe_youtube(
#     url=demo_url,
#     model_size="tiny",
#     output_path=os.path.join(outputs_dir, 'talks_transcripts.jsonl')
# )
# print(f"\nFirst 300 chars of transcript:")
# print(demo_transcript[:300])

print("▶️  Demo cell commented out to prevent accidental long-running downloads.")
print("   Uncomment lines above and run to test, OR go straight to TODO 3 below.")

🧪 Experiment 3 (Demo): ASR with faster-whisper

Demo video: https://www.youtube.com/watch?v=rBCqOTEfxvg
Using model: tiny (fastest, ~1-3 min on CPU for first 3 min of audio)

Note: First run downloads the tiny model (~75MB). Subsequent runs use cache.

✓ Downloaded (pytubefix): "Attention is all you need; Attentional Neural Network Models | Łukasz Kaiser | Masterclass" (2903s)
Transcribing with faster-whisper (tiny)...
✓ Transcribed 976 segments
  Detected language: en
  Transcript length: 38049 chars
✓ Appended record to: ../outputs/talks_transcripts.jsonl

First 300 chars of transcript:
Okay, so you've heard a great talk just before about how to use neural networks to translate. And this has been quite a big thing in all of natural language processing. So people have been doing a lot of stuff in natural language processing, parsing, translation, finding entities, anything towards m
▶️  Demo cell commented out to prevent accidental long-running downloads.
   Uncomment lines above and 

### 🎯 TODO 3: Transcribe Your Own Videos

In [4]:
# TODO 3: Choose 3 YouTube videos and transcribe them
#
# Choose videos relevant to YOUR project domain, or interesting talks you've watched.
# Keep each video under 5 minutes to avoid long runtimes.

my_urls = [
    # "https://www.youtube.com/watch?v=...",   # Video 1 title: _____
    # "https://www.youtube.com/watch?v=...",   # Video 2 title: _____
    # "https://www.youtube.com/watch?v=...",   # Video 3 title: _____
]

print("=" * 65)
print("🎯 TODO 3: My Video Transcriptions")
print("=" * 65)
print()

if my_urls:
    for i, url in enumerate(my_urls, 1):
        print(f"[{i}/{len(my_urls)}] Processing: {url}")
        transcript = transcribe_youtube(
            url=url,
            model_size="tiny",
            output_path=os.path.join(outputs_dir, 'talks_transcripts.jsonl')
        )
        print(f"  Preview: {transcript[:200]}...")
        print()
else:
    print("⚠️  No URLs provided.")
    print("   Add YouTube URLs to my_urls above and re-run.")
    print()
    print("   Suggestions (short NLP/ML talks):")
    print("   - Yannic Kilcher: https://www.youtube.com/@YannicKilcher")
    print("   - Andrej Karpathy: https://www.youtube.com/@AndrejKarpathy")
    print("   - 3Blue1Brown: https://www.youtube.com/@3blue1brown")

todo3_reflection = """
[YOUR REFLECTION HERE]

- Which videos did you transcribe? Why did you choose them?
- How accurate was the tiny model on domain-specific vocabulary (technical terms)?
- What would you get with 'base' or 'small' model size instead?
- Where does transcribed speech data fit into LLM pretraining? What's unique about it
  compared to web text? (Think: informal language, spoken grammar, filler words)
"""
print()
print(todo3_reflection)

🎯 TODO 3: My Video Transcriptions

⚠️  No URLs provided.
   Add YouTube URLs to my_urls above and re-run.

   Suggestions (short NLP/ML talks):
   - Yannic Kilcher: https://www.youtube.com/@YannicKilcher
   - Andrej Karpathy: https://www.youtube.com/@AndrejKarpathy
   - 3Blue1Brown: https://www.youtube.com/@3blue1brown


[YOUR REFLECTION HERE]

- Which videos did you transcribe? Why did you choose them?
- How accurate was the tiny model on domain-specific vocabulary (technical terms)?
- What would you get with 'base' or 'small' model size instead?
- Where does transcribed speech data fit into LLM pretraining? What's unique about it
  compared to web text? (Think: informal language, spoken grammar, filler words)



## Summary & Reflection

In [7]:
# Summary of outputs created
print("=" * 65)
print("📊 NOTEBOOK 04 OUTPUTS SUMMARY")
print("=" * 65)
print()

output_files = {
    'arxiv_clean.json':     os.path.join(outputs_dir, 'arxiv_clean.json'),
    'pdf_ocr/ (folder)':   os.path.join(outputs_dir, 'pdf_ocr'),
    'talks_transcripts.jsonl': os.path.join(outputs_dir, 'talks_transcripts.jsonl'),
}

for name, path in output_files.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ (not yet created)'
    size   = ''
    if exists and os.path.isfile(path):
        sz = os.path.getsize(path)
        size = f'({sz/1024:.1f} KB)'
    print(f"  {status} {name} {size}")

print()

# Use defaults if TODO cells were skipped
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'
_todo3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed yet]'
_cat   = MY_CATEGORY if 'MY_CATEGORY' in dir() else '[not set]'
_papers = str(len(my_papers)) if 'my_papers' in dir() else '[not run]'
_urls  = str(len(my_urls)) if 'my_urls' in dir() else '0'

full_reflection = f"""
### Section 1 — Web Scraping (TODO 1)

Category: {_cat}
Papers collected: {_papers}

{_todo1}

---

### Section 2 — PDF OCR (TODO 2)

{_todo2}

---

### Section 3 — ASR Transcription (TODO 3)

Videos transcribed: {_urls}

{_todo3}
"""

reflection_file = append_to_reflection(
    notebook="04",
    section_title="Data Collection Pipeline",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"✅ Reflection saved: {reflection_file}")


📊 NOTEBOOK 04 OUTPUTS SUMMARY

  ✅ arxiv_clean.json (31.3 KB)
  ✅ pdf_ocr/ (folder) 
  ✅ talks_transcripts.jsonl (39.6 KB)

✅ Reflection saved: ../outputs/homework_reflection.md


## ✅ Notebook 04 Complete!

**What you accomplished:**
- ✅ Scraped arXiv papers using Trafilatura (web scraping)
- ✅ Extracted text from PDFs using pdfplumber and Tesseract OCR
- ✅ Transcribed YouTube audio using faster-whisper ASR
- ✅ Saved raw collected data to `outputs/`

**Key concepts:**
- Web scraping ≠ web crawling — Trafilatura extracts *clean* content, not raw HTML
- pdfplumber (text layer) >> Tesseract (OCR) when text layer is available
- faster-whisper runs 4× faster than OpenAI Whisper with same quality
- All three data sources have quality and diversity trade-offs

**Next:** Open **Notebook 05: Data Cleaning Pipeline** 🧹

The raw data you collected needs extensive cleaning before it can be used for training. That's what we tackle next!